# Day 8 — Inheritance & Polymorphism
### Python for Data Science · Module 1 · Topic 1.8

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | Inheritance — reusing a class you already wrote | 25 min |
| 2 | Overriding and `super()` | 25 min |
| 3 | Polymorphism | 20 min |
| 4 | Multiple inheritance, and when to stop | 15 min |
| 5 | Mini build: an `Employee` hierarchy | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **Everything today rests on `self`.** If yesterday's `self` felt shaky, say so now —
> the next 90 minutes will be hard otherwise.
>
> **Where this shows up:** scikit-learn has hundreds of models, and every one answers
> `.fit()` and `.predict()`. That is exactly what you learn today.

---
## 0. Recap of Day 7

In [ ]:
class Student:
    school = "BU"                      # class attribute - shared

    def __init__(self, name):
        self.name = name               # instance attribute - per object

    def greet(self):
        return f"Hi, I am {self.name}"

    def __str__(self):
        return f"Student({self.name})"


ravi = Student("Ravi")
print(ravi.greet())
print(Student.greet(ravi))     # the same call - self is the object before the dot
print(ravi)                    # __str__

---
# 1. Inheritance

## 1.1 The problem it solves

In [1]:
# WITHOUT inheritance - two classes that are mostly identical
class Student:
    def __init__(self, name, age):
        self.name = name
        self.age  = age
    def greet(self):
        return f"Hi, I am {self.name}"

class Teacher:
    def __init__(self, name, age):    # identical
        self.name = name              # identical
        self.age  = age               # identical
    def greet(self):                  # identical
        return f"Hi, I am {self.name}"

print(Student("Ravi", 20).greet())
print(Teacher("Mr Rao", 40).greet())

Hi, I am Ravi
Hi, I am Mr Rao


In [3]:
# WITH inheritance - write the shared part once
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age  = age

    def greet(self):
        return f"Hi, I am {self.name}"


class Student(Person):     # inherits everything
    pass

class Teacher(Person):     # inherits everything
    pass


print(Student("Ravi", 20).greet())
print(Teacher("Mr Rao", 40).greet())

Hi, I am Ravi
Hi, I am Mr Rao


This is **Day 3's lesson one level up**. Functions stopped you repeating a calculation;
inheritance stops you repeating a whole class. Fix `greet()` once in `Person` and every
child is fixed.

Note the `pass` — Day 2's placeholder, used properly. A child that adds nothing is still
useful, because it names a type.

## 1.2 What the child gets

```
              Person
        name · age · greet()
           /            \
     Student           Teacher
     + school          + subject
```

The child gets **every method**, **every attribute `__init__` sets**, and
**every class attribute** — for free.

The arrow points from child to parent: *"a Student IS A Person"*.

In [5]:
class Person:
    species = "human"                # class attribute

    def __init__(self, name):
        self.name = name

    def greet(self):
        return f"Hi, I am {self.name}"


class Student(Person):
    school = "BU"    
    
class Professor(Person):
    subject = "Maths"                # adds one of its own


ravi = Student("Ravi")
rao = Professor("Rao")

print(ravi.name)        # from Person.__init__
print(ravi.greet())     # from Person
print(ravi.species)     # Person's class attribute
print(ravi.school)      # Student's own


print(rao.name)
print(rao.greet())
print(rao.species)
print(rao.subject)

Ravi
Hi, I am Ravi
human
BU
Rao
Hi, I am Rao
human
Maths


## 1.3 `isinstance` and `issubclass`

In [ ]:
print(isinstance(ravi, Student))      # True
print(isinstance(ravi, Person))       # True - also! It really IS a Person
print(issubclass(Student, Person))    # True
print(issubclass(Person, Student))    # False - the other way round

print(Student.__bases__)              # who the parents are

`isinstance(ravi, Person)` being `True` is the key insight. A `Student` is not merely
*similar* to a `Person` — it **is** one. That is why any code written to accept a `Person`
will happily accept a `Student`, which is the whole basis of section 3.

---
# 2. Overriding and `super()`

## 2.1 Overriding — the child's version wins

In [9]:
class Person:
    def greet(self):
        return "Hi, I am a person"

class Student(Person):
    def greeth(self):                    # same name - replaces the parent's
        return "Hi, I am a student"


print(Person().greet())
print(Student().greeth())
print(Student().greet())

Hi, I am a person
Hi, I am a student
Hi, I am a person


### How Python finds a method

1. Is it in `Student`?  → use it
2. If not, is it in `Person`? → use it
3. Still not found → `AttributeError`

Python stops at the **first** match, so the child always wins.

> This is **Day 3's LEGB rule with a different chain**. Names are found by walking a chain
> and stopping at the first hit. For variables: Local → Enclosing → Global → Built-in.
> For attributes: the object → its class → its parent → up to `object`.
> A child's method shadows the parent's exactly as a local variable shadows a global.

## 2.2 `super()` — extending instead of replacing

In [11]:
class Person:
    def __init__(self, name):
        self.name = name
    def greet(self):
        return f"Hi, I am {self.name}"


# Overriding throws the parent's work away
class Student1(Person):
    def greet(self):
        return "Hi, I study at BU"          # the name is gone


# super() runs the parent's version first, then adds to it
class Student2(Person):
    def greet(self):
        base = super().greet()
        return base + ", I study at BU"


print(Student1("Ravi").greet())
print(Student2("Ravi").greet())

Hi, I study at BU
Hi, I am Ravi, I study at BU


## 2.3 ⚠️ The line you must not forget: `super().__init__(...)`

Writing your own `__init__` **replaces** the parent's completely.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age  = age


# BROKEN - the child's __init__ replaced the parent's, so name is never set
class Student(Person):
    def __init__(self, name, age, school):
        self.school = school          # forgot super().__init__


ravi = Student("Ravi", 20, "BU")
print("school:", ravi.school)         # this works...
try:
    print(ravi.name)                  # ...but this does not
except AttributeError as e:
    print("AttributeError:", e)

In [ ]:
# CORRECT - hand off to the parent FIRST, then add what is new
class Student(Person):
    def __init__(self, name, age, school):
        super().__init__(name, age)   # parent's setup
        self.school = school          # then your own


ravi = Student("Ravi", 20, "BU")
print(ravi.name, ravi.age, ravi.school)

> **Call `super().__init__()` first, then add what is new.** If you call it last, it may
> overwrite attributes you have just set.
>
> Without it, the first thing that reads `self.name` dies with an `AttributeError` —
> often far away from the real cause.

---
# 3. Polymorphism — one call, many behaviours

In [ ]:
class Animal:
    def speak(self):
        return "..."

class Dog(Animal):
    def speak(self): return "Woof"

class Cat(Animal):
    def speak(self): return "Meow"

class Cow(Animal):
    def speak(self): return "Moo"


for a in [Dog(), Cat(), Cow(), Dog()]:
    print(a.speak())

# The loop NEVER asks what kind of animal it has.

Woof
Meow
Moo
Woof


### Compare with the version that checks types

In [ ]:
# WITHOUT polymorphism - a branch per class
def speak_bad(a):
    if isinstance(a, Dog):
        return "Woof"
    elif isinstance(a, Cat):
        return "Meow"
    elif isinstance(a, Cow):
        return "Moo"
    # ...and one more branch every time a new animal appears

print(speak_bad(Dog()))

**The real benefit is what happens when you add a new class.**

Adding a `Sheep` means writing the `Sheep` class and *nothing else* — the loop already
works. In the `if`/`elif` version you must find and edit every branching block in the
codebase, and any one you miss becomes a bug.

Polymorphism moves the decision out of your code and into the objects themselves.

In [ ]:
# Proof: add a new class, change nothing else
class Sheep(Animal):
    def speak(self): return "Baa"

for a in [Dog(), Cat(), Cow(), Sheep()]:      # same loop as before
    print(a.speak())

## 3.1 Duck typing — Python does not even require inheritance

In [ ]:
class Dog:                 # no parent
    def speak(self): return "Woof"

class Robot:               # no parent, no relation to Dog at all
    def speak(self): return "Beep"


for x in [Dog(), Robot()]:
    print(x.speak())       # works anyway

*"If it walks like a duck and quacks like a duck..."*

Python never checks an object's type before calling a method. It simply tries. If the object
has a `speak()` method, the call works — the family tree is irrelevant.

This is why `len()` works on a string, a list, a dict and your own class: anything that
defines `__len__`.

### So when is inheritance still worth it?

| | |
|---|---|
| **Shared code** | the parent holds the logic all children need |
| **A shared type** | `isinstance(x, Animal)` becomes meaningful |
| **A contract** | the parent documents what every child must provide |

Duck typing gives you polymorphism *without* inheritance. Inheritance gives you shared
implementation on top. Use inheritance when the children genuinely share code — not merely
because they share a method name.

---
# 4. Multiple inheritance

## 4.1 Two parents, and the search order

In [13]:
class Swimmer:
    def move(self): return "swims"

class Flyer:
    def move(self): return "flies"

class Duck(Swimmer, Flyer):      # two parents
    pass


print(Duck().move())             # "swims" - Swimmer came FIRST
print(Duck.__mro__)              # the full search order

swims
(<class '__main__.Duck'>, <class '__main__.Swimmer'>, <class '__main__.Flyer'>, <class 'object'>)


Python builds one flat search order — the **MRO**, or Method Resolution Order — from the
class and its parents, left to right, and stops at the first match.

### ⚠️ It gets confusing quickly

With two parents the order is easy to read. With four parents that themselves have parents,
working out which `move()` runs becomes a research exercise — and every reader of your code
has to do it too.

Most professional Python uses single inheritance almost exclusively. **Recognise** multiple
inheritance when you meet it in a library; think hard before writing it yourself.

## 4.2 When inheritance is the wrong tool

In [ ]:
class Engine:
    def start(self): return "vroom"


# WRONG - is a Car a kind of Engine? No.
class Car1(Engine):
    pass

print(Car1().start())      # it WORKS, but the claim is false

In [ ]:
# RIGHT - composition. The car HAS an engine.
class Car2:
    def __init__(self):
        self.engine = Engine()

    def start(self):
        return self.engine.start()


print(Car2().start())

# Now the relationship is honest, and you could swap in an ElectricEngine
# without touching any family tree.

> ### The test, before you write `class Child(Parent):`
>
> Say it out loud:
> - *"a Student **IS A** Person"* — true, so inherit
> - *"a Car **IS AN** Engine"* — false, so give the car an engine instead
>
> If the sentence sounds wrong to an ordinary English speaker, the inheritance is wrong
> too — however convenient the code reuse looks.

---
# 5. Putting it together — an `Employee` hierarchy

In [ ]:
class Employee:
    """Anyone on the payroll."""

    def __init__(self, name, salary):
        self.name   = name
        self.salary = salary

    def pay(self):                          # children override this
        return self.salary / 12

    def __str__(self):
        return f"{self.name}: {self.pay():.0f}/month"


class Manager(Employee):
    def __init__(self, name, salary, bonus):
        super().__init__(name, salary)      # parent setup FIRST
        self.bonus = bonus                  # then what is new

    def pay(self):                          # EXTENDS the parent
        return super().pay() + self.bonus


class Intern(Employee):
    def pay(self):                          # REPLACES the parent
        return 1000


staff = [
    Employee("Ravi", 60000),
    Manager("Sara", 90000, 500),
    Intern("Amit", 0),
]

for person in staff:                        # polymorphism
    print(person)

In [ ]:
# Checks worth running
print("Manager is an Employee:", isinstance(staff[1], Employee))
print("Manager pay = 7500 + 500 =", staff[1].pay())
print("Intern ignores salary   :", staff[2].pay())

The deepest point is the last one: `Employee.__str__` calls `self.pay()`, and when `self`
is a `Manager`, **the child's `pay()` runs**. A parent method calling `self.something()`
gets the child's override — polymorphism working *inside* the parent class.

---
# 6. Recap — the twelve things to remember

1. `class Child(Parent)` inherits every method and attribute.
2. A child method with the same name **overrides** the parent's.
3. Python searches the child first, then up the chain.
4. `super()` calls the parent's version instead of replacing it.
5. A child `__init__` **must** call `super().__init__(...)` first.
6. `isinstance(child_obj, Parent)` is `True` — it really is one.
7. Polymorphism: one loop, many classes, no type checks.
8. Adding a new class needs no change to existing loops.
9. Duck typing: a shared method name is enough, no parent needed.
10. Two parents are searched left to right — see `__mro__`.
11. A parent calling `self.method()` gets the **child's** version.
12. *"IS A"* → inherit.  *"HAS A"* → composition.

---

### 📝 Now open **`Day8_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Build a `Shape` parent with `Circle` and `Square` children, each with `area()`.
- Extend Day 7's `BankAccount` into a `SavingsAccount` that adds interest.
- Find one place in your own code where inheritance would be the wrong choice.

### Next class — Topic 1.9: File I/O
Reading and writing text and CSV files, the `with` statement, and file modes.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*